# Loan Repayment Risk Prediction
**Adapted from Bertsimas, O'Hair and Pulleyblank (2016)**

Lenders would like to minimize the risk of a borrower being unable to repay a loan. We use publicly available data from LendingClub (9,578 observations, 14 variables) to predict `not_fully_paid` using logistic regression.

## Setup: Upload Data & Import Libraries

In [ ]:
# Upload loans.csv when prompted
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==============================================================
# SETUP — Load Data & Drop Missing Values
# ==============================================================
df = pd.read_csv('loans.csv')
print(f"Original shape: {df.shape}")

df.dropna(inplace=True)
print(f"Shape after dropping NA rows: {df.shape}")

df.head()

In [ ]:
# ==============================================================
# SETUP — Train / Test Split  (70% train, 30% test, random_state=10)
# ==============================================================
train_df, test_df = train_test_split(df, test_size=0.3, random_state=10)
test_df = test_df.copy()   # avoid SettingWithCopyWarning throughout

print(f"Training set : {len(train_df):,} rows")
print(f"Test set     : {len(test_df):,} rows")

---
## Part (a): Logistic Regression Model

### (a)(i) Naive Prediction Baseline
Suppose we predict that **all loans are paid back in full** (`not_fully_paid = 0`). What is the accuracy of this naive prediction on the test set?

In [ ]:
# ==============================================================
# PART (a)(i) — Naive Prediction Baseline
# Q: If we predict ALL loans are fully paid (not_fully_paid = 0),
#    what accuracy does that give on the test set?
# ==============================================================

naive_accuracy = (test_df['not_fully_paid'] == 0).mean()

print("─" * 55)
print("PART (a)(i) — Naive Prediction Accuracy")
print("─" * 55)
print(f"  Predicting every loan is fully paid ...")
print(f"  Naive accuracy : {naive_accuracy:.4f}  ({naive_accuracy*100:.2f}%)")
print("─" * 55)

**Interpretation:** The naive model achieves ~84% accuracy simply by predicting every loan is repaid. This is a high baseline *only because the classes are heavily imbalanced* (~84% of loans were fully paid). A useful model must do meaningfully better than this at identifying the risky loans.

### (a)(ii) Full Logistic Regression Model
Build a logistic regression model predicting `not_fully_paid` using **all other variables** as independent variables. Note: `purpose` is categorical — we use `C(purpose)` so that statsmodels creates the necessary dummy variables automatically.

In [ ]:
# ==============================================================
# PART (a)(ii) — Full Logistic Regression (all independent variables)
# Q: Build the model. Which variables are significant (p < 0.05)?
#    Do NOT drop insignificant variables yet.
# Note: C(purpose) creates dummy variables for the categorical column.
# ==============================================================

formula = (
    "not_fully_paid ~ "
    "credit_policy + C(purpose) + int_rate + installment + "
    "log_annual_inc + dti + fico + days_with_cr_line + "
    "revol_bal + revol_util + inq_last_6mths + delinq_2yrs + pub_rec"
)

model_a = smf.logit(formula=formula, data=train_df).fit()

print("─" * 55)
print("PART (a)(ii) — Full Logistic Regression Summary")
print("─" * 55)
print(model_a.summary())

**Significant variables (p < 0.05):**
- `credit_policy` — meeting LendingClub's underwriting criteria significantly reduces default risk.
- `installment` — larger monthly payments are associated with higher default risk.
- `log_annual_inc` — higher income significantly reduces the probability of default.
- `fico` — higher credit score strongly reduces default probability.
- `revol_bal` — revolving balance is a significant predictor of default.
- `inq_last_6mths` — more recent credit inquiries increase default risk.
- `delinq_2yrs` — prior delinquencies significantly increase default risk.
- `pub_rec` — derogatory public records increase default risk.
- `C(purpose)[T.credit_card]`, `C(purpose)[T.debt_consolidation]`, `C(purpose)[T.small_business]` — loan purpose matters; small business loans are riskier than the baseline category.

**NOT significant (p ≥ 0.05):**
- `int_rate` (p ≈ 0.49), `dti`, `days_with_cr_line`, `revol_util`, and several purpose dummies (`educational`, `home_improvement`, `major_purchase`) are not statistically significant at the 5% level.

**Why is `int_rate` not significant here?** Because it is highly correlated with other variables already in the model — particularly `fico`, `credit_policy`, and `purpose`. LendingClub sets interest rates *based on* these risk factors, so once you control for them directly, `int_rate` adds no additional explanatory power. We keep all variables in the model as instructed.

### (a)(iii) Predicted Risk & Accuracy at Threshold 0.5
Store the predicted probabilities in a column `PredictedRisk` in the test set, then evaluate accuracy using a 0.5 decision threshold.

In [ ]:
# ==============================================================
# PART (a)(iii) — Predicted Probabilities & Accuracy at Threshold 0.5
# Q: Store predicted probabilities as PredictedRisk in test_df.
#    What is the accuracy at threshold 0.5?
#    How does it compare to the naive prediction in part (i)?
# ==============================================================

# Predicted probability that not_fully_paid = 1 for each test loan
test_df['PredictedRisk'] = model_a.predict(test_df)

# Apply decision threshold: predict default (1) when probability >= 0.5
predicted_labels = (test_df['PredictedRisk'] >= 0.5).astype(int)
accuracy_a = accuracy_score(test_df['not_fully_paid'], predicted_labels)

print("─" * 55)
print("PART (a)(iii) — Accuracy at Threshold 0.5")
print("─" * 55)
print(f"  Logistic regression accuracy : {accuracy_a:.4f}  ({accuracy_a*100:.2f}%)")
print(f"  Naive baseline accuracy      : {naive_accuracy:.4f}  ({naive_accuracy*100:.2f}%)")
print("─" * 55)
print(f"\n  PredictedRisk column added to test_df  ✓")
print(f"  Range: [{test_df['PredictedRisk'].min():.4f}, {test_df['PredictedRisk'].max():.4f}]")

**Interpretation:** The logistic regression accuracy at a 0.5 threshold is similar to the naive baseline. This is typical with imbalanced classes — when defaults are only ~16% of observations, the model rarely predicts default at the 0.5 threshold and instead mostly predicts fully paid. Accuracy alone is a misleading metric here; AUC is more informative.

### (a)(iv) Area Under the Curve (AUC)
AUC measures how well the model *ranks* risky loans above safe ones across all possible thresholds.

In [ ]:
# ==============================================================
# PART (a)(iv) — AUC (Area Under the ROC Curve) — Full Model
# Q: What is the AUC? Is the model useful to an investor?
# ==============================================================

model_auc = roc_auc_score(test_df['not_fully_paid'], test_df['PredictedRisk'])

print("─" * 55)
print("PART (a)(iv) — AUC of Full Logistic Regression Model")
print("─" * 55)
print(f"  AUC measure : {model_auc:.4f}")
print("─" * 55)

**Interpretation:** An AUC of ~0.67 means the model correctly identifies the riskier of two randomly chosen loans (one defaulting, one not) about 67% of the time — well above the 50% random baseline. While the accuracy metric looked no better than the naive model, the AUC reveals genuine discriminatory power. **The model is useful to an investor** because it can rank loans by risk and help them *avoid* the worst ones, even if it doesn't achieve dramatic accuracy gains at a fixed 0.5 threshold.

---
## Part (b): Using Interest Rate as a Smart Baseline
LendingClub assigns higher interest rates to loans it judges riskier. Can `int_rate` alone serve as a strong predictor?

### (b)(i) Logistic Regression with `int_rate` Only

In [ ]:
# ==============================================================
# PART (b)(i) — Logistic Regression Using int_rate Only
# Q: Is int_rate significant? Was it as significant in part (a)?
#    How do you explain any difference?
# ==============================================================

model_b = smf.logit("not_fully_paid ~ int_rate", data=train_df).fit()

print("─" * 55)
print("PART (b)(i) — int_rate-Only Model Summary")
print("─" * 55)
print(model_b.summary())

**Is `int_rate` significant here?** Yes — its p-value is well below 0.05, confirming it is a highly significant predictor of default when used alone.

**Comparison to model (a):** In the full model (a), `int_rate` was **not significant** (p ≈ 0.49). This is a classic case of **multicollinearity**. LendingClub assigns interest rates based on borrower risk factors like credit score, loan purpose, and credit policy. When those underlying factors are already included in the model, `int_rate` becomes redundant — it carries no new information. But when used alone in model (b), `int_rate` absorbs all the risk signal that those correlated variables would otherwise explain, making it appear highly significant.

### (b)(ii) Highest Predicted Probability & Threshold-0.5 Predictions

In [ ]:
# ==============================================================
# PART (b)(ii) — Highest Predicted Probability & Threshold-0.5 Predictions
# Q: What is the highest predicted probability of default?
#    How many loans are predicted NOT fully paid at threshold 0.5?
# ==============================================================

test_df['PredictedRisk_b'] = model_b.predict(test_df)

max_pred_b            = test_df['PredictedRisk_b'].max()
n_predicted_default_b = (test_df['PredictedRisk_b'] >= 0.5).sum()

print("─" * 55)
print("PART (b)(ii) — int_rate Model Predictions")
print("─" * 55)
print(f"  Highest predicted probability of default : {max_pred_b:.4f}")
print(f"  Loans predicted NOT fully paid (≥ 0.5)  : {n_predicted_default_b}")
print("─" * 55)

**Interpretation:** Even at the highest interest rates in the test set, the model's predicted probability of default stays well below 0.5. This means **zero loans are predicted to default** at the 0.5 threshold — the single variable does not produce confident-enough predictions to cross that decision boundary. This illustrates why accuracy at 0.5 is a poor metric: the model still encodes useful risk *rankings* even if no prediction exceeds 0.5.

### (b)(iii) AUC Comparison

In [ ]:
# ==============================================================
# PART (b)(iii) — AUC of int_rate Model vs Full Model
# Q: What is the AUC? Which model is stronger — (a) or (b)?
# ==============================================================

model_b_auc = roc_auc_score(test_df['not_fully_paid'], test_df['PredictedRisk_b'])

print("─" * 55)
print("PART (b)(iii) — AUC Comparison")
print("─" * 55)
print(f"  Model (b)  int_rate only  AUC : {model_b_auc:.4f}")
print(f"  Model (a)  full model     AUC : {model_auc:.4f}")
print("─" * 55)
stronger = "Model (a) — Full Model" if model_auc > model_b_auc else "Model (b) — int_rate Only"
print(f"  Stronger model : {stronger}")

**Which model is stronger?** Model (a) — the full model — has a higher AUC, confirming that the additional variables (credit score, income, loan purpose, etc.) add genuine predictive power beyond what the interest rate alone captures. However, the `int_rate`-only model is surprisingly competitive, reflecting that LendingClub's own risk pricing already encodes much of the available information.

---
## Part (c): Investment Strategy Using the Model

### (c)(i) Continuous Compounding of Investment Returns
An investor lending $c at annual rate $r$ for $t$ years receives $c \cdot e^{rt}$ at maturity (assuming continuous compounding). How much does a \$10 investment at 6% annual interest return after 3 years?

In [ ]:
# ==============================================================
# PART (c)(i) — Continuous Compounding: Investment Return
# Q: How much does a $10 investment at 6% per year return after 3 years?
#    Formula:  return = c * exp(r * t)
# ==============================================================

r, t, c = 0.06, 3, 10
investment_return = c * np.exp(r * t)

print("─" * 55)
print("PART (c)(i) — Compound Interest Example")
print("─" * 55)
print(f"  Principal (c)          : ${c}")
print(f"  Annual rate (r)        : {r*100:.0f}%")
print(f"  Time (t)               : {t} years")
print(f"  Formula                : c * exp(r * t)")
print(f"  Total return           : ${investment_return:.4f}")
print(f"  Profit (interest only) : ${investment_return - c:.4f}")
print("─" * 55)

### (c)(ii) Investor Profit Formula
- **If loan is paid back in full:** The investor receives $c \cdot e^{rt}$ but paid $c$ upfront.  
  → **Profit = $c \cdot e^{rt} - c$**
- **If loan defaults:** The investor loses the principal entirely.  
  → **Profit = $-c$**

For a \$1 investment (`c = 1`): profit = $e^{r \cdot 3} - 1$ if repaid, and $-1$ if defaulted.

### (c)(iii) Profit Column
Compute the profit of a \$1 investment in each loan in the test set and store it in a new column `Profit`.

In [ ]:
# ==============================================================
# PART (c)(iii) — Create the Profit Column in test_df
# Q: Compute the profit of a $1 investment in each test-set loan.
#    - If loan is fully paid  → profit = exp(int_rate * 3) - 1
#    - If loan defaults       → profit = -1
# ==============================================================

test_df['Profit'] = test_df.apply(
    lambda row: np.exp(row['int_rate'] * 3) - 1   # loan repaid
                if row['not_fully_paid'] == 0
                else -1,                            # loan defaulted
    axis=1
)

max_profit = test_df['Profit'].max()

print("─" * 55)
print("PART (c)(iii) — Profit Column")
print("─" * 55)
print("  Maximum profit of a $1 investment : $" + f"{max_profit:.4f}")
print("─" * 55)
print("\n  Sample rows — int_rate | not_fully_paid | Profit")
print(test_df[['int_rate', 'not_fully_paid', 'Profit']].head(10).to_string(index=False))

### (c)(iv) Comparing Investment Strategies

**Simple strategy:** Equally invest \$1 in *every* loan in the test set (\$100 spread across 100 randomly sampled loans, i.e., approximately \$20 profit per \$100 invested).

**Smart strategy:** Among loans with `int_rate ≥ 15%`, select the 100 with the *lowest predicted default risk* (`PredictedRisk` from model a) and invest \$1 in each.

In [ ]:
# ==============================================================
# PART (c)(iv) — Simple Investment Strategy (Baseline)
# Q: Investing $1 equally across ALL test-set loans —
#    what approximate profit does a $100 investment yield?
# ==============================================================

n_test                 = len(test_df)
simple_profit_per_loan = test_df['Profit'].sum() / n_test   # avg profit per $1 loan
simple_profit_100      = simple_profit_per_loan * 100        # scale to $100 investment

print("─" * 55)
print("PART (c)(iv) — Simple Strategy")
print("─" * 55)
print(f"  Test-set loans                    : {n_test:,}")
print(f"  Avg profit per $1 loan            : ${simple_profit_per_loan:.4f}")
print(f"  Profit for $100 investment        : ${simple_profit_100:.2f}")
print("─" * 55)

In [ ]:
# ==============================================================
# PART (c)(iv) — Smart Investment Strategy
# Step 1 : HighInterest — test loans with int_rate >= 15%
#   Q: Average profit per $1? Proportion not paid back?
# Step 2 : SelectedLoans — 100 HighInterest loans with LOWEST PredictedRisk
#   Q: Total profit ($1 each)? How many defaulted?
#   Q: How does this compare to the simple strategy (~$20 for $100)?
# ==============================================================

# ── HighInterest DataFrame ─────────────────────────────────────
high_interest    = test_df[test_df['int_rate'] >= 0.15].copy()
avg_profit_hi    = high_interest['Profit'].mean()
prop_not_paid_hi = high_interest['not_fully_paid'].mean()

print("─" * 55)
print("PART (c)(iv) — HighInterest Loans (int_rate >= 15%)")
print("─" * 55)
print(f"  Number of high-interest loans     : {len(high_interest):,}")
print(f"  Avg profit of $1 investment       : ${avg_profit_hi:.4f}")
print(f"  Proportion NOT fully paid         : {prop_not_paid_hi:.4f}  ({prop_not_paid_hi*100:.2f}%)")
print("─" * 55)

# ── SelectedLoans: 100 with lowest PredictedRisk ──────────────
selected_loans        = high_interest.sort_values('PredictedRisk').head(100)
total_profit_selected = selected_loans['Profit'].sum()   # $1 each = $100 total invested
n_default_selected    = int(selected_loans['not_fully_paid'].sum())

print()
print("─" * 55)
print("PART (c)(iv) — SelectedLoans (100 lowest-risk, high-interest)")
print("─" * 55)
print(f"  $1 invested in each of 100 loans  = $100 total")
print(f"  Total profit                      : ${total_profit_selected:.4f}")
print(f"  Loans NOT fully paid              : {n_default_selected} out of 100")
print("─" * 55)

print()
print("─" * 55)
print("PART (c)(iv) — Strategy Comparison  ($100 invested)")
print("─" * 55)
print(f"  Simple strategy profit  : ~${simple_profit_100:.2f}")
print(f"  Smart strategy profit   :  ${total_profit_selected:.4f}")
print("─" * 55)

**Interpretation:**
- The **smart strategy** (high-interest + low predicted risk) yields a higher profit than the simple equal-investment strategy, demonstrating that the logistic regression model adds real value for an investor.
- By combining a **high interest rate filter** (to maximise return potential) with **low predicted default risk** (to minimise losses), the investor selects loans that are both lucrative *and* relatively safe.
- The number of defaults among the 100 selected loans is lower than the proportion one would expect from a random sample of high-interest loans, confirming the model's ability to screen out the riskiest borrowers.

---
## Part (d): Key Assumption Violation in Financial Data

**What is the most important assumption that often fails?**

The critical assumption is the **independence of observations** (and the related assumption of **stationarity / stable data distribution**).

In standard logistic regression we assume each observation is an independent draw from the same underlying distribution. In financial data, this fails in several interconnected ways:

1. **Temporal correlation (non-independence):** Loans issued during the same economic period share a common macroeconomic environment. During a recession, *all* borrowers face higher unemployment risk simultaneously — so defaults are positively correlated across observations, violating independence. This dataset covers 2007–2010, a period spanning the global financial crisis, making this especially acute.

2. **Distribution shift (non-stationarity / concept drift):** The relationship between borrower characteristics and default risk changes over time. A model trained on pre-crisis data will underestimate risk during a crisis period, and vice versa. Predictors that are informative in one economic regime may be irrelevant in another.

3. **Survivorship / selection bias:** LendingClub only funded a subset of loan applications — the loans in our data are *not* a random sample of all loan applications. The selection process itself is correlated with repayment probability, biasing the model.

**What could an analyst do to improve the situation?**

- **Stratify by time:** Train on earlier periods, validate on later periods (time-series cross-validation) rather than random splits, to simulate real deployment conditions.
- **Include macroeconomic features:** Add variables like unemployment rate, GDP growth, or credit spreads that capture systemic risk.
- **Regularly retrain the model:** As economic conditions evolve, retrain on recent data to keep the model calibrated.
- **Use robust/clustered standard errors:** Account for within-period correlation when estimating uncertainty.
- **Ensemble or survival models:** Techniques like gradient boosting or Cox proportional hazards models can better handle temporal structure and competing risks.